# 2.3 Create Metadata Dataset Splits

Purpose:
- Load the stroke prediction metadata dataset
- Create stratified train/validation/test splits
- Preserve severe class imbalance across splits

In [1]:
import pandas as pd

from pathlib import Path
import sys

sys.path.insert(0, str(Path.cwd().parent))

from rural_stroke_assist.config import (
    METADATA_DATA_DIR,
    METADATA_SPLIT_MANIFEST_PATH,
)
from rural_stroke_assist.preprocessing.splitting import (
    add_stratified_split,
    summarize_split_balance,
)

In [2]:
csv_files = list(METADATA_DATA_DIR.rglob("*.csv"))
metadata_df = pd.read_csv(csv_files[0])

metadata_df.head()

,id,gender,age,hypertension,heart_disease,ever_married,work_type,Residence_type,avg_glucose_level,bmi,smoking_status,stroke
0,9046,Male,67.0,0,1,Yes,Private,Urban,228.69,36.6,formerly smoked,1
1,51676,Female,61.0,0,0,Yes,Self-employed,Rural,202.21,NaN,never smoked,1
2,31112,Male,80.0,0,1,Yes,Private,Rural,105.92,32.5,never smoked,1
3,60182,Female,49.0,0,0,Yes,Private,Urban,171.23,34.4,smokes,1
4,1665,Female,79.0,1,0,Yes,Self-employed,Rural,174.12,24.0,never smoked,1


In [3]:
metadata_df["stroke"].value_counts(normalize=True) * 100

stroke
0    95.127202
1     4.872798
Name: proportion, dtype: float64

In [4]:
metadata_split_df = add_stratified_split(
    df=metadata_df,
    label_column="stroke",
    train_size=0.70,
    val_size=0.15,
    test_size=0.15,
    random_state=42,
)

metadata_split_df.head()

,id,gender,age,hypertension,heart_disease,ever_married,work_type,Residence_type,avg_glucose_level,bmi,smoking_status,stroke,split
0,9046,Male,67.0,0,1,Yes,Private,Urban,228.69,36.6,formerly smoked,1,train
1,51676,Female,61.0,0,0,Yes,Self-employed,Rural,202.21,NaN,never smoked,1,train
2,31112,Male,80.0,0,1,Yes,Private,Rural,105.92,32.5,never smoked,1,train
3,60182,Female,49.0,0,0,Yes,Private,Urban,171.23,34.4,smokes,1,train
4,1665,Female,79.0,1,0,Yes,Self-employed,Rural,174.12,24.0,never smoked,1,train


In [5]:
split_summary = summarize_split_balance(
    df=metadata_split_df,
    split_column="split",
    label_column="stroke",
)

split_summary

,split,stroke,count,percentage
0,test,0,729,95.05
1,test,1,38,4.95
2,train,0,3403,95.14
3,train,1,174,4.86
4,val,0,729,95.17
5,val,1,37,4.83


In [6]:
METADATA_SPLIT_MANIFEST_PATH.parent.mkdir(parents=True, exist_ok=True)

metadata_split_df.to_csv(METADATA_SPLIT_MANIFEST_PATH, index=False)

print("Saved metadata split manifest to:", METADATA_SPLIT_MANIFEST_PATH)

Saved metadata split manifest to: C:\Users\Asus\Downloads\Uni_work\Grad-Project\rural-stroke-assist\data\processed\metadata_split_manifest.csv


## Split Decision

The metadata dataset was split using stratified sampling because the positive stroke class is very small.

This preserves the target distribution across train, validation, and test sets.

Limitation:
This dataset represents general stroke risk factors, not acute stroke triage. It is kept as a contextual metadata baseline, not as the main clinical evidence source.